# Exercise 4 - Your event: fit, feature importance, and compare with the tutorial event

**Time budget:** ~30 min &nbsp;|&nbsp; **Reference:** notebook 04, Tasks 3-4 &nbsp;|&nbsp; **Uses your Exercise-1 batch**

Your Exercise-1 batch has produced some `results.csv` rows by now. Stand up the surrogate for
*your* event on whatever has completed, read a slice, rank the parameters by their effect on
arrival time, and compare with the **tutorial event** (2017-09-06 from notebook 04).

You are given the grid and a `show_slice(...)` plotter; the fitting and the analysis are yours.

**By the end you can:** build a surrogate for your event and see how its sensitivity structure differs from the
tutorial event.

In [ ]:
# --- Google Colab bootstrap (no-op locally) ---
import sys, os

if "google.colab" in sys.modules:
    REPO_URL = os.environ.get("CONECAST_REPO", "https://github.com/georgemilosh/conecast")
    REPO_DIR = "/content/conecast"
    if not os.path.isdir(REPO_DIR):
        os.system(f"git clone --depth 1 {REPO_URL} {REPO_DIR}")
    os.chdir(REPO_DIR)
    try:
        import sunpy, huxt, wsaplus  # noqa: F401
        print("Colab bootstrap complete; cwd =", os.getcwd())
    except ModuleNotFoundError:
        print("Installing sunpy + WSA+ + HUXt (one-time, ~2 min)...")
        os.system("pip install -q sunpy wsaplus")
        os.system("pip install -q "
                  "'huxt @ git+https://github.com/University-of-Reading-Space-Science/HUXt'")
        print("Done - restarting the runtime. When it reconnects, RUN THIS CELL AGAIN.")
        os.kill(os.getpid(), 9)
else:
    print("Not in Colab - using the local checkout.")

In [ ]:
from pathlib import Path
import sys
cwd = Path.cwd().resolve()
BASE_DIR = cwd if (cwd / "scripts").exists() else cwd.parent
SCRIPT_DIR = BASE_DIR / "scripts"
if str(SCRIPT_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPT_DIR))
print("BASE_DIR =", BASE_DIR)

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import gp_huxt_surrogate as gp
PARAM_NAMES = gp.PARAM_NAMES
DATA_ROOT = BASE_DIR / "data_dir" / "sw"; GP_ROOT = BASE_DIR / "runs" / "gp_surrogate"

EVENT = ...   # the event you provisioned in Exercise 1 (an id from data_dir/events.csv)
res = pd.read_csv(GP_ROOT / EVENT / "results.csv")
done = res.loc[res["status"] == "completed"]
hit_rate = done["hit"].map(lambda v: str(v).lower() in ("true", "1", "yes")).mean()
print(f"{EVENT}: {len(done)} completed runs so far, hit rate {round(hit_rate, 3)}")
if len(done) < 30:
    print("Few rows so far - let the Exercise-1 batch finish, then re-run this cell.")

ctx = gp.load_huxt_context(EVENT, DATA_ROOT); theta0 = ctx["theta0"]
span = np.array([1.0, 30.0, 20.0, 40.0, 0.25 * theta0[4]]); low, high = theta0 - span, theta0 + span

xi, yi, n = 1, 4, 60                                   # longitude vs v
gx = np.linspace(low[xi], high[xi], n); gy = np.linspace(low[yi], high[yi], n)
XX, YY = np.meshgrid(gx, gy)
GRID = np.tile(theta0, (n * n, 1)); GRID[:, xi] = XX.ravel(); GRID[:, yi] = YY.ravel()

def show_slice(P, arrival_mean):
    P = np.asarray(P).reshape(n, n); A = np.asarray(arrival_mean).reshape(n, n)
    fig, ax = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
    c0 = ax[0].contourf(XX, YY, P, levels=20, cmap="viridis"); ax[0].contour(XX, YY, P, [0.5], colors="white")
    ax[0].set_title(f"P(hit) - {EVENT}"); fig.colorbar(c0, ax=ax[0])
    c1 = ax[1].contourf(XX, YY, A, levels=20, cmap="magma"); ax[1].set_title("arrival mean [hr] (hits)")
    fig.colorbar(c1, ax=ax[1])
    for a in ax: a.set_xlabel("longitude"); a.plot(theta0[xi], theta0[yi], "w*", ms=12)
    ax[0].set_ylabel("v"); plt.tight_layout()

## Task 4.1 - Fit the surrogate and read the slice

In [ ]:
# ------------------------------ YOUR CODE HERE ------------------------------

# 1. Fit the GP bundle for EVENT:  bundle = gp.train_models(event, output_root, test_fraction, random_state)
# 2. P = gp.predict_hit_probability(bundle, GRID)
# 3. arrival_mean, _ = gp.predict_arrival(bundle, GRID)
#    then MASK arrival_mean where P < 0.5 (np.where(..., np.nan)) - arrival is only physical
#    where the CME hits (the no-hit-region fix from notebook 04, Task 4a).
# 4. show_slice(P, arrival_mean)


## Task 4.2 - Which launch parameter drives arrival time?

In [ ]:
# ------------------------------ YOUR CODE HERE ------------------------------

# Build a finite-difference sensitivity of arrival time at the seed (tutorial 04, Task 4b):
# For each parameter i in range(5):
#   - step = 1% of that parameter's range:  max((high[i] - low[i]) * 0.01, 1e-6)
#   - nudge theta0 up and down by step, clipped into [low[i], high[i]]
#   - central-difference the GP arrival mean:  (arrival(plus) - arrival(minus)) / (plus[i] - minus[i])
# Units differ per parameter, so also compute per-range leverage: gradient * (high[i] - low[i])  [hr].
# Print or bar-plot the ranking by leverage.


## Task 4.3 - Interpret, and compare with the tutorial event

1. Is your event **hit-prone** or **miss-prone**? Does the `P(hit)` slice match the prediction
   you wrote in Exercise 1?
2. Which parameter has the largest **per-range leverage** on arrival time? Does that make
   physical sense for your event's geometry and speed?
3. **Compare two events.** Tutorial notebook 04 builds a 300-run surrogate for **2017-09-06**
   (a wide, fast, near-central CME). If you've run it, its batch is at
   `runs/gp_surrogate/2017-09-06/results.csv` - load it the same way and contrast: is your event
   more or less hit-prone, and does the *same* parameter dominate arrival time? (For a fast limb
   event, longitude should matter more than for the near-central tutorial one.)

### Stretch
- Provision a *third* event (repeat Exercise 1 with a new `EVENT`) for a three-way comparison.
- Apply the Exercise-3 compound threshold to *your* event and compare hit rates.